# 2. Strategy Development

向量化快速回测 -> 验证信号方向 -> 搬入 trader.py -> Jmerle 回测

In [ ]:
import sys
sys.path.insert(0, '.')

from utils.dataio import load_prices_wide, load_trades
from utils.orderbook import compute_wall_mid, compute_spread
import polars as pl
import numpy as np

In [ ]:
ROUND = 0
PRODUCT = 'EMERALDS'

## 向量化回测：Taker 策略

不模拟撮合，只检验 "如果我在 ask < fair 时吃单" 能赚多少。

In [ ]:
# 加载两天数据
days = [-2, -1]
all_spreads = []
for day in days:
    pw = load_prices_wide(ROUND, day).filter(pl.col('product') == PRODUCT)
    sp = compute_spread(pw).with_columns(pl.lit(day).alias('day_label'))
    all_spreads.append(sp)

spreads = pl.concat(all_spreads)
print(f'Total rows: {len(spreads)}')
spreads.head()

In [ ]:
def vectorized_taker_pnl(spreads: pl.DataFrame, fair_col='wall_mid'):
    """简易 taker PnL：每步如果 best_ask < fair 就买，best_bid > fair 就卖。"""
    df = spreads.sort('timestamp')
    fair = df[fair_col].to_numpy()
    bid_wall = df['bid_wall_price'].to_numpy()
    ask_wall = df['ask_wall_price'].to_numpy()
    
    # 简易 PnL：吃单价 vs fair 的差值累加
    buy_edge = fair - ask_wall  # positive = profitable buy
    sell_edge = bid_wall - fair  # positive = profitable sell
    
    buy_pnl = np.where(buy_edge > 0, buy_edge, 0).cumsum()
    sell_pnl = np.where(sell_edge > 0, sell_edge, 0).cumsum()
    total_pnl = buy_pnl + sell_pnl
    
    return total_pnl

pnl = vectorized_taker_pnl(spreads)
print(f'Total taker edge: {pnl[-1]:.1f}')

## Jmerle Backtester

In [ ]:
from backtest.runner import run_all_days

# 确保 trader.py 已更新后运行
# results = run_all_days(round_num=ROUND)